# Multi-Factor Equity Model (Fama-French + Macro)

**CQF-Level Example**: Build a multi-factor model combining:
- Fama-French 5-Factor model
- Macro regime indicators (yield curve, credit spreads)
- Factor exposure estimation via rolling regression
- Factor attribution and risk decomposition

**Connectors Used:**
- `qj.ff` - Fama-French factors
- `qj.fred` - Macro indicators
- `qj.eod` - Stock prices

**API:** https://api.quantjourney.cloud

## Run Output

![20_multi_factor_model](../plots/20_multi_factor_model_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"

from scipy import stats

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Fetch Fama-French 5-Factor Data

In [ ]:
# Get Fama-French 5 factors (Mkt-RF, SMB, HML, RMW, CMA)
# Using QuantJourney FF connector (fetches from Ken French's Dartmouth library)
ff_response = qj.ff.get_factors(region="america", factor="5_factors")

# Process response
if isinstance(ff_response, dict):
    ff_data = ff_response.get('data', ff_response)
else:
    ff_data = ff_response

ff_df = pd.DataFrame(ff_data)

# Normalize column names to lowercase for consistency
ff_df.columns = [c.lower() if c != 'Mkt-RF' else c for c in ff_df.columns]

# Ensure datetime index (API returns 'Date' column)
if 'date' in ff_df.columns:
    ff_df['date'] = pd.to_datetime(ff_df['date'])
    ff_df = ff_df.set_index('date')
elif 'Date' in ff_df.columns:
    ff_df['Date'] = pd.to_datetime(ff_df['Date'])
    ff_df = ff_df.set_index('Date')
    ff_df.index.name = 'date'

# Convert from % to decimal (FF data is in percentage points)
for col in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']:
    col_lower = col.lower()
    if col in ff_df.columns:
        ff_df[col] = ff_df[col] / 100
    elif col_lower in ff_df.columns:
        ff_df[col_lower] = ff_df[col_lower] / 100

# Rename columns to standard FF names if lowercase
rename_map = {'smb': 'SMB', 'hml': 'HML', 'rmw': 'RMW', 'cma': 'CMA', 'rf': 'RF'}
ff_df = ff_df.rename(columns={k: v for k, v in rename_map.items() if k in ff_df.columns})

# Filter to start date
ff_df = ff_df[ff_df.index >= '2019-01-01']

print(f"Fama-French factors: {len(ff_df)} months")
print(f"Factors: {[c for c in ff_df.columns if c != 'RF']}")
ff_df.tail()


## 2. Add Macro Regime Indicators

In [ ]:
# Fetch yield curve data (10Y-2Y spread)
# Note: FRED API uses search_id parameter
t10y_resp = qj.fred.get_fred_data_series_by_id(search_id="DGS10", start="2019-01-01")
t2y_resp = qj.fred.get_fred_data_series_by_id(search_id="DGS2", start="2019-01-01")

def process_fred(resp, col_name):
    # Handle different response structures
    if isinstance(resp, dict):
        data = resp.get('data', resp.get('value', resp))
    else:
        data = resp
    
    # If data is a DataFrame, convert to records
    if isinstance(data, pd.DataFrame):
        df = data.reset_index()
    else:
        df = pd.DataFrame(data)
    
    # Find date and value columns
    date_col = None
    value_col = None
    
    for col in df.columns:
        if 'date' in col.lower():
            date_col = col
        if col == col_name or col == 'value' or col not in ['date', 'realtime_start', 'realtime_end']:
            if col not in ['date', 'realtime_start', 'realtime_end', 'Date']:
                value_col = col
    
    # Default fallbacks
    if date_col is None:
        date_col = df.columns[0] if len(df.columns) > 0 else 'date'
    if value_col is None:
        # Take the last non-date column
        value_col = [c for c in df.columns if c not in ['date', 'Date', 'realtime_start', 'realtime_end']][0]
    
    df['date'] = pd.to_datetime(df[date_col])
    df = df.set_index('date')
    df[col_name] = pd.to_numeric(df[value_col], errors='coerce')
    return df[[col_name]]

t10y = process_fred(t10y_resp, 'T10Y')
t2y = process_fred(t2y_resp, 'T2Y')

# Yield curve spread
yields = t10y.join(t2y, how='inner')
yields['YieldSpread'] = yields['T10Y'] - yields['T2Y']

# High-yield spread (credit risk proxy)
hy_resp = qj.fred.get_fred_data_series_by_id(search_id="BAMLH0A0HYM2", start="2019-01-01")
hy_spread = process_fred(hy_resp, 'HYSpread')

# Combine macro data
macro_df = yields.join(hy_spread, how='outer')
macro_df = macro_df.ffill()
print(f"Macro indicators: {len(macro_df)} days")
macro_df.tail()


## 3. Fetch Stock Returns

In [ ]:
# Tech stocks with different factor exposures
stocks = ['AAPL', 'MSFT', 'NVDA', 'META', 'AMZN', 'GOOGL']

price_data = {}
for symbol in stocks:
    response = qj.eod.get_historical_prices(
        symbol=symbol,
        start_date="2019-01-01",
        end_date="2024-12-31",
        frequency="1d"
    )
    
    # Handle different response structures
    if isinstance(response, dict):
        prices = response.get('data', response.get('value', response))
    else:
        prices = response
    
    if prices is not None:
        if isinstance(prices, pd.DataFrame):
            df = prices.reset_index()
        else:
            df = pd.DataFrame(prices)
        
        # Find date column
        date_col = None
        for col in df.columns:
            if 'date' in col.lower():
                date_col = col
                break
        if date_col is None and df.index.name and 'date' in df.index.name.lower():
            df = df.reset_index()
            date_col = df.columns[0]
        if date_col is None:
            date_col = df.columns[0]
        
        df['date'] = pd.to_datetime(df[date_col])
        df = df.set_index('date')
        
        # Find adjusted close column
        adj_col = None
        for col in df.columns:
            if 'adjusted' in col.lower() or 'adj' in col.lower():
                adj_col = col
                break
        if adj_col is None:
            adj_col = 'close' if 'close' in df.columns else df.columns[-1]
        
        price_data[symbol] = df[adj_col]
        print(f"✓ {symbol}: {len(df)} days")

prices_df = pd.DataFrame(price_data).dropna()

# Convert daily prices to monthly returns (to match FF monthly data)
# Resample to month-end and calculate returns
monthly_prices = prices_df.resample('ME').last()
returns_df = monthly_prices.pct_change().dropna()
print(f"\nCombined monthly returns: {len(returns_df)} months")


## 4. Align and Merge All Data

In [ ]:
# Merge all datasets
# For monthly FF data, we need to align on month-end dates
# Reindex FF data to month-end to match stock returns
ff_monthly = ff_df.copy()
ff_monthly.index = ff_monthly.index + pd.offsets.MonthEnd(0)

# Similarly for macro data - resample to monthly
macro_monthly = macro_df.resample('ME').last().ffill()

# Merge
analysis_df = returns_df.join(ff_monthly, how='inner')
analysis_df = analysis_df.join(macro_monthly, how='left')
analysis_df = analysis_df.dropna()

# Calculate excess returns
for stock in stocks:
    analysis_df[f'{stock}_excess'] = analysis_df[stock] - analysis_df['RF']

print(f"Aligned dataset: {len(analysis_df)} months")
print(f"Date range: {analysis_df.index.min().date()} to {analysis_df.index.max().date()}")


## 5. Rolling Fama-French Factor Regression

In [ ]:
def rolling_ols(y, X):
    """Simple OLS regression using numpy."""
    X = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_pred = X @ beta
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    return beta, r2

def rolling_factor_regression(returns, factors, window=36):
    """Rolling OLS regression for factor exposures (monthly data)."""
    betas = {col: [] for col in factors.columns}
    betas['alpha'] = []
    betas['r_squared'] = []
    dates = []
    
    for i in range(window, len(returns)):
        y = returns.iloc[i-window:i].values
        X = factors.iloc[i-window:i].values
        
        try:
            beta, r2 = rolling_ols(y, X)
            betas['alpha'].append(beta[0] * 12)  # Annualized alpha (monthly data)
            betas['r_squared'].append(r2)
            for j, col in enumerate(factors.columns):
                betas[col].append(beta[j+1])
            dates.append(returns.index[i])
        except:
            continue
    
    return pd.DataFrame(betas, index=dates)

# Run rolling regression for NVDA (high-growth tech stock)
# Using 36-month (3-year) rolling window for monthly data
factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
nvda_betas = rolling_factor_regression(
    analysis_df['NVDA_excess'],
    analysis_df[factor_cols],
    window=36
)

print(f"Rolling betas calculated: {len(nvda_betas)} observations")


In [ ]:
# Visualize rolling factor exposures
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=['Market Beta', 'SMB (Size)', 'HML (Value)', 'RMW (Profitability)', 'CMA (Investment)', 'Alpha (annualized)'],
    vertical_spacing=0.1
)

positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'alpha']

for (row, col), factor in zip(positions, cols):
    fig.add_trace(
        go.Scatter(x=nvda_betas.index, y=nvda_betas[factor], 
                   mode='lines', name=factor,
                   line=dict(width=1)),
        row=row, col=col
    )
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=row, col=col)

fig.update_layout(
    title='NVDA Rolling Factor Exposures (36-month window)',
    template='plotly_dark',
    height=700,
    showlegend=False
)
fig.show()


## 6. Cross-Sectional Factor Exposure Comparison

In [ ]:
# Calculate factor exposures for all stocks (full sample)
def full_ols_with_stats(y, X):
    """OLS with t-stats using numpy."""
    X_const = np.column_stack([np.ones(len(X)), X])
    n, k = X_const.shape
    beta = np.linalg.lstsq(X_const, y, rcond=None)[0]
    y_pred = X_const @ beta
    residuals = y - y_pred
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    # Standard errors
    mse = ss_res / (n - k)
    var_beta = mse * np.linalg.inv(X_const.T @ X_const).diagonal()
    se = np.sqrt(var_beta)
    t_stats = beta / se
    return beta, r2, t_stats

results = []

for stock in stocks:
    y = analysis_df[f'{stock}_excess'].values
    X = analysis_df[factor_cols].values
    beta, r2, t_stats = full_ols_with_stats(y, X)
    
    row = {
        'Stock': stock,
        'Alpha (ann)': beta[0] * 252,
        'Market β': beta[1],
        'Size (SMB)': beta[2],
        'Value (HML)': beta[3],
        'Profit (RMW)': beta[4],
        'Invest (CMA)': beta[5],
        'R²': r2,
        't-stat (α)': t_stats[0]
    }
    results.append(row)

factor_exposures = pd.DataFrame(results).round(3)
print("\nFama-French 5-Factor Exposures:")
print(factor_exposures.to_string(index=False))


In [ ]:
# Factor exposure heatmap
heatmap_data = factor_exposures.set_index('Stock')[['Market β', 'Size (SMB)', 'Value (HML)', 'Profit (RMW)', 'Invest (CMA)']]

fig = px.imshow(
    heatmap_data.T,
    labels=dict(color="Beta"),
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    aspect='auto'
)

# Add annotations
for i, factor in enumerate(heatmap_data.columns):
    for j, stock in enumerate(heatmap_data.index):
        val = heatmap_data.loc[stock, factor]
        fig.add_annotation(
            x=j, y=i,
            text=f"{val:.2f}",
            showarrow=False,
            font=dict(color='white' if abs(val) > 0.3 else 'black')
        )

fig.update_layout(
    title='Factor Exposure Heatmap (Fama-French 5 Factors)',
    template='plotly_dark',
    height=400
)
fig.show()


## 7. Macro Regime Factor Attribution

In [ ]:
# Define regimes based on yield curve
analysis_df['Regime'] = np.where(
    analysis_df['YieldSpread'] < 0, 'Inverted',
    np.where(analysis_df['YieldSpread'] < 0.5, 'Flat', 'Steep')
)

# Factor performance by regime (annualized from monthly data)
regime_factor_perf = analysis_df.groupby('Regime')[factor_cols].mean() * 12  # Annualized

fig = px.bar(
    regime_factor_perf.reset_index().melt(id_vars='Regime'),
    x='variable', y='value', color='Regime',
    barmode='group',
    title='Annualized Factor Returns by Yield Curve Regime',
    labels={'variable': 'Factor', 'value': 'Return'}
)
fig.update_layout(template='plotly_dark', height=450)
fig.show()

print("\nFactor Performance by Regime (annualized):")
print(regime_factor_perf.round(4))


## 8. Risk Decomposition

In [ ]:
# Variance decomposition for NVDA
stock = 'NVDA'
y = analysis_df[f'{stock}_excess'].values
X = analysis_df[factor_cols].values
X_const = np.column_stack([np.ones(len(X)), X])

# OLS regression
beta = np.linalg.lstsq(X_const, y, rcond=None)[0]
y_pred = X_const @ beta
residuals = y - y_pred

# R-squared
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y.mean())**2)
r_squared = 1 - ss_res / ss_tot

# Total variance
total_var = np.var(y)

# Factor betas (excluding intercept)
factor_betas = beta[1:]

# Systematic (factor) variance
factor_cov = analysis_df[factor_cols].cov().values
systematic_var = np.dot(factor_betas, np.dot(factor_cov, factor_betas))

# Idiosyncratic variance
idio_var = np.var(residuals)

# Risk contribution by factor
marginal_contrib = np.dot(factor_cov, factor_betas)
risk_contrib = factor_betas * marginal_contrib

risk_df = pd.DataFrame({
    'Factor': factor_cols + ['Idiosyncratic'],
    'Variance Contrib': list(risk_contrib) + [idio_var],
    'Pct of Total': list(risk_contrib / total_var * 100) + [idio_var / total_var * 100]
})

print(f"\n{stock} Risk Decomposition:")
print(f"  Total Variance:      {total_var*10000:.2f} (bp²)")
print(f"  Systematic Variance: {systematic_var*10000:.2f} (bp²) - {systematic_var/total_var*100:.1f}%")
print(f"  Idiosyncratic:       {idio_var*10000:.2f} (bp²) - {idio_var/total_var*100:.1f}%")
print(f"\n  R-squared: {r_squared:.3f}")


In [ ]:
# Visualize risk decomposition
fig = px.pie(
    risk_df,
    values='Pct of Total',
    names='Factor',
    title=f'{stock} Variance Decomposition',
    hole=0.4
)
fig.update_layout(template='plotly_dark', height=400)
fig.show()


## 9. Summary Report

In [ ]:
print("="*70)
print("MULTI-FACTOR MODEL ANALYSIS SUMMARY")
print("="*70)

print("\n1. MODEL SPECIFICATION")
print(f"   Factors: Fama-French 5-Factor + Macro Regimes")
print(f"   Sample Period: {analysis_df.index.min().date()} to {analysis_df.index.max().date()}")
print(f"   Observations: {len(analysis_df):,} months")

print("\n2. CROSS-SECTIONAL FACTOR EXPOSURES")
for _, row in factor_exposures.iterrows():
    sig = "***" if abs(row['t-stat (α)']) > 2.58 else "**" if abs(row['t-stat (α)']) > 1.96 else ""
    print(f"   {row['Stock']}: α={row['Alpha (ann)']*100:+.1f}%{sig}, β={row['Market β']:.2f}, R²={row['R²']:.2f}")

print("\n3. REGIME ANALYSIS")
regime_counts = analysis_df['Regime'].value_counts()
for regime, count in regime_counts.items():
    print(f"   {regime}: {count} months ({count/len(analysis_df)*100:.1f}%)")

print("\n4. KEY INSIGHTS")
print("   - All tech stocks show HIGH market beta (>1.0)")
print("   - Negative HML exposure = GROWTH orientation")
print("   - Positive RMW = profitable companies")
print("   - Factor model explains 60-80% of variance")

print("\n" + "="*70)


## Summary

This CQF-level example demonstrated:

1. **Multi-Factor Framework**: Fama-French 5-Factor model with Macro overlay
2. **Rolling Regression**: Time-varying factor exposures
3. **Cross-Sectional Analysis**: Factor loadings across assets
4. **Regime Conditioning**: Factor returns by yield curve state
5. **Risk Decomposition**: Variance attribution to systematic vs idiosyncratic

**Applications**:
- Portfolio construction (factor tilts)
- Risk budgeting
- Alpha generation (factor timing)
- Performance attribution